In [2]:
import pandas as pd
from pathlib import Path

In [3]:
# data path
DATA_PATH = Path(
    r"C:\Users\jchmo\Downloads\dataset_football_matches\big5"
)

In [4]:
# dictionary of the league names
league_names = {
    "E0": "Premier League",
    "D1": "Bundesliga",
    "I1": "Serie A",
    "SP1": "La Liga",
    "F1": "Ligue 1"
}

In [5]:
# load all the data
dataframes = []

for file in sorted(DATA_PATH.glob("*.csv")):

    # eg. E0_2122
    file_name = file.stem

    # split the name to season and league
    league_code, season_code = file_name.split("_")

    # league name
    league = league_names[league_code]

    # 2122 -> 2021/22
    start_year = 2000 + int(season_code[:2])
    end_year = 2000 + int(season_code[2:])

    season = f"{start_year}/{str(end_year)[-2:]}"

    # load CSV
    try:
        df_temp = pd.read_csv(file, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df_temp = pd.read_csv(file, encoding="cp1252")

    # delete empty columns
    df_temp.columns = df_temp.columns.str.strip()

    # add info about league and season
    df_temp["Season"] = season
    df_temp["League"] = league
    df_temp["LeagueCode"] = league_code

    # save the data to dataframe
    dataframes.append(df_temp)

# concatenate all dataframes
df = pd.concat(
    dataframes,
    ignore_index=True,
    sort=False
)

In [6]:
# Pracujeme s kópiou pôvodného datasetu
df_ml = df.copy()

# Dátum prevedieme na datetime
df_ml["Date"] = pd.to_datetime(
    df_ml["Date"],
    dayfirst=True,
    errors="coerce"
)

# Odstránime zápasy bez základných údajov
df_ml = df_ml.dropna(
    subset=["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
)

# Zoradíme zápasy chronologicky
df_ml = df_ml.sort_values(
    ["Date"]
).reset_index(drop=True)

# Vytvoríme históriu zápasov každého tímu
home = df_ml[["Date", "League", "HomeTeam", "FTHG", "FTAG"]].copy()
home.columns = ["Date", "League", "Team", "GoalsFor", "GoalsAgainst"]

away = df_ml[["Date", "League", "AwayTeam", "FTAG", "FTHG"]].copy()
away.columns = ["Date", "League", "Team", "GoalsFor", "GoalsAgainst"]

# Spojíme domáce a vonkajšie zápasy do jednej histórie
history = pd.concat([home, away], ignore_index=True)
history = history.sort_values(["Date"]).reset_index(drop=True)

# Body za zápas
history["Points"] = (
    (history["GoalsFor"] > history["GoalsAgainst"]).astype(int) * 3
    + (history["GoalsFor"] == history["GoalsAgainst"]).astype(int)
)

history["Win"] = (
    history["GoalsFor"] > history["GoalsAgainst"]
).astype(int)

# Pre každý tím vytvoríme štatistiky z predchádzajúcich 5 zápasov
grouped = history.groupby(["League", "Team"], sort=False)

history["PointsLast5"] = grouped["Points"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).sum()
)

history["WinsLast5"] = grouped["Win"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).sum()
)

history["GoalsForAvgLast5"] = grouped["GoalsFor"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

history["GoalsAgainstAvgLast5"] = grouped["GoalsAgainst"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

# Počet zápasov v histórii, ktoré tím už odohral
history["MatchesBefore"] = grouped.cumcount()

# Pripravíme históriu na pripojenie k domácemu a hosťujúcemu tímu
feature_cols = [
    "Date", "League", "Team",
    "PointsLast5", "WinsLast5",
    "GoalsForAvgLast5", "GoalsAgainstAvgLast5",
    "MatchesBefore"
]

team_features = history[feature_cols].copy()

# Domáce features
home_features = team_features.rename(columns={
    "Team": "HomeTeam",
    "PointsLast5": "home_points_last5",
    "WinsLast5": "home_wins_last5",
    "GoalsForAvgLast5": "home_goals_avg_last5",
    "GoalsAgainstAvgLast5": "home_conceded_avg_last5",
    "MatchesBefore": "home_matches_before"
})

# Hosťujúce features
away_features = team_features.rename(columns={
    "Team": "AwayTeam",
    "PointsLast5": "away_points_last5",
    "WinsLast5": "away_wins_last5",
    "GoalsForAvgLast5": "away_goals_avg_last5",
    "GoalsAgainstAvgLast5": "away_conceded_avg_last5",
    "MatchesBefore": "away_matches_before"
})

# Pripojíme historické štatistiky k jednotlivým zápasom
df_ml = df_ml.merge(
    home_features,
    on=["Date", "League", "HomeTeam"],
    how="left",
    validate="many_to_one"
)

df_ml = df_ml.merge(
    away_features,
    on=["Date", "League", "AwayTeam"],
    how="left",
    validate="many_to_one"
)

# Zatiaľ vynecháme zápasy, pri ktorých nemáme žiadnu históriu
df_ml = df_ml[
    (df_ml["home_matches_before"] > 0)
    & (df_ml["away_matches_before"] > 0)
].copy()

print(df_ml[[
    "Date", "League", "HomeTeam", "AwayTeam",
    "home_points_last5", "away_points_last5",
    "home_goals_avg_last5", "away_goals_avg_last5",
    "FTR"
]].head(10))


         Date   League   HomeTeam     AwayTeam  home_points_last5  \
12 2021-08-13  Ligue 1    Lorient       Monaco                1.0   
22 2021-08-14  Ligue 1      Lille         Nice                1.0   
23 2021-08-14  Ligue 1   Paris SG   Strasbourg                3.0   
33 2021-08-15  Ligue 1   Clermont       Troyes                3.0   
34 2021-08-15  Ligue 1      Reims  Montpellier                1.0   
35 2021-08-15  Ligue 1       Lens   St Etienne                1.0   
36 2021-08-15  Ligue 1  Marseille     Bordeaux                3.0   
39 2021-08-15  Ligue 1     Nantes         Metz                1.0   
40 2021-08-15  Ligue 1     Angers         Lyon                3.0   
41 2021-08-15  Ligue 1      Brest       Rennes                1.0   

    away_points_last5  home_goals_avg_last5  away_goals_avg_last5 FTR  
12                1.0                   1.0                   1.0   H  
22                1.0                   3.0                   0.0   A  
23                0.0   

In [8]:
# Skontroluj názvy sezón
print(sorted(df_ml["Season"].astype(str).unique()))

# Štyri sezóny na tréning, posledná na test
train_seasons = ["2122", "2223", "2324", "2425"]
test_season = "2526"

train_df = df_ml[
    df_ml["Season"].astype(str).isin(train_seasons)
].copy()

test_df = df_ml[
    df_ml["Season"].astype(str) == test_season
].copy()

# Vyberieme iba features známe pred zápasom
features = [
    "home_points_last5",
    "away_points_last5",
    "home_wins_last5",
    "away_wins_last5",
    "home_goals_avg_last5",
    "away_goals_avg_last5",
    "home_conceded_avg_last5",
    "away_conceded_avg_last5",
]

X_train = train_df[features]
y_train = train_df["FTR"]

X_test = test_df[features]
y_test = test_df["FTR"]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

['2021/22', '2022/23', '2023/24', '2024/25', '2025/26']
Train: (0, 8)
Test: (0, 8)


In [9]:
# 4 sezóny na tréning
train_seasons = ["2021/22", "2022/23", "2023/24", "2024/25"]

# Posledná sezóna na test
test_season = "2025/26"

train_df = df_ml[
    df_ml["Season"].astype(str).isin(train_seasons)
].copy()

test_df = df_ml[
    df_ml["Season"].astype(str) == test_season
].copy()

features = [
    "home_points_last5",
    "away_points_last5",
    "home_wins_last5",
    "away_wins_last5",
    "home_goals_avg_last5",
    "away_goals_avg_last5",
    "home_conceded_avg_last5",
    "away_conceded_avg_last5"
]

X_train = train_df[features]
y_train = train_df["FTR"]

X_test = test_df[features]
y_test = test_df["FTR"]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print("\nTrain seasons:")
print(sorted(train_df["Season"].astype(str).unique()))

print("\nTest seasons:")
print(sorted(test_df["Season"].astype(str).unique()))


Train: (7080, 8)
Test: (1747, 8)

Train seasons:
['2021/22', '2022/23', '2023/24', '2024/25']

Test seasons:
['2025/26']


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Vytvorenie modelu
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

# Trénovanie
rf_model.fit(X_train, y_train)

# Predikcia na testovacích dátach
y_pred = rf_model.predict(X_test)

# Vyhodnotenie
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred), 4))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=["H", "D", "A"],
    target_names=["Home Win", "Draw", "Away Win"],
    zero_division=0
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=["H", "D", "A"]))


Accuracy: 0.4803
Balanced accuracy: 0.4089

Classification report:
              precision    recall  f1-score   support

    Home Win       0.50      0.79      0.61       768
        Draw       0.21      0.01      0.02       444
    Away Win       0.45      0.43      0.44       535

    accuracy                           0.48      1747
   macro avg       0.39      0.41      0.36      1747
weighted avg       0.41      0.48      0.41      1747


Confusion matrix:
[[605  10 153]
 [312   4 128]
 [300   5 230]]
